1. 문서의 내용으 읽는다
2. 문서를 쪼갠다.
  - 토큰 수 초과로 답변을 생성하지 못할 수 있고
  - 문서가 길면(Input) 답변 생성이 오래걸림
3. 쪼갠 문서를 임배딩 -> 백터 데이터베이스에 저장
4. 질문이 있을 때, 백터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [1]:
# %pip install --upgrade --quiet  docx2txt langchain-community


In [2]:
# 문서 쪼개기 Text Splitter

# Character Text Splitter : 구분자를 하나만 넣을 수 있다.
# Recursive Character Text Splitter : 구분자를 여러개 넣을 수 있다.

# %pip install -qU langchain-text-splitters







In [3]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # 문서를 쪼갤 때 하나의 청크가 가질 수 있는 청크의 수
    chunk_size=1500,
    # 문서를 곂치는 설정
    chunk_overlap=200
)

loader = Docx2txtLoader("./tax.docx")
document_list = loader.load_and_split(text_splitter=text_splitter)

In [4]:
len(document_list)

193

In [5]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings


load_dotenv()

embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')


In [6]:
# %pip install langchain-chroma

In [ ]:
from langchain_chroma import Chroma


# 데이터베이스를 처음 생성할 때

# database = Chroma.from_documents(
#     documents=document_list,
#     embedding=embedding,
#     persist_directory='./chroma',
#     collection_name='chroma-tax'
#     )

# 생성된 데이터베이스를 사용할 때

database = Chroma(
    collection_name='chroma-tax',
    embedding_function=embedding,
    persist_directory='./chroma'
)


In [8]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query, k=3)

In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')



In [10]:
prompt = f"""
- 당신은 최고의 한국 소득세 전문가 입니다.
- [Context]를 참고해서 사용자의 질문에 답변해주세요.

[Context]
{retrieved_docs}

Question : {query}
"""

In [11]:
ai_message = llm.invoke(prompt)

ai_message.content




'연봉 5천만 원인 직장인의 소득세를 계산하기 위해서는 여러 가지 요소를 고려해야 합니다. 한국의 소득세는 누진세 구조로 이루어져 있으며, 연봉의 크기에 따라 세율이 달라집니다. 먼저, 근로소득공제를 반영한 후 남은 금액에 대해 소득세를 계산해야 합니다. 이 외에도 각종 세액공제와 감면 항목이 추가로 적용될 수 있습니다.\n\n일단 기본적인 계산을 위해, 2023년 기준의 세율표를 참고로 설명해 보겠습니다. 일반적으로 소득세는 연 소득 금액에 따라 다음과 같은 기본 누진세율 구조를 가집니다 (대략적인 수치, 변경 가능성 있음):\n\n1. 1,200만 원 이하: 6%\n2. 1,200만 원 초과 ~ 4,600만 원 이하: 15%\n3. 4,600만 원 초과 ~ 8,800만 원 이하: 24%\n4. 8,800만 원 초과 ~ 1억 5천만 원 이하: 35%\n5. 1억 5천만 원 초과: 38%\n\n예를 들어, 연봉 5천만 원인 경우, 다음과 같이 단순화하여 근로소득공제 및 기본 세율을 적용한 계산 순서로 소득세를 계산할 수 있습니다(단순 예시):\n\n1. 근로소득공제를 적용 (대략적인 경우로 세율에 따라 5,000만 원에서 공제)\n2. 남은 금액에 대해 누진세율 적용: \n    - 1,200만 원 이하 구간: 1,200만 원 x 6% \n    - 1,200만 원 초과 ~ 4,600만 원 이하 구간: (4,600-1,200) x 15% \n    - 4,600만 원 초과 구간의 나머지 금액: (5,000-4,600) x 24%\n\n이렇게 각각의 세율구간 별로 산출된 세액을 합산하여 총 소득세 납부액을 계산합니다. 실제 세액은 연말정산 시 공제 및 감면 혜택이 추가로 반영되어 최종적으로 더욱 정확히 결정됩니다.\n\n보다 정확한 계산을 위해서는 근로소득공제, 기본공제 및 추가적 세액공제 항목을 모두 반영해야 하므로, 이를 토대로 국세청 홈택스의 연말정산 소득세 계산기를 이용하거나 세무 전문가와 상담하는 것이 좋습니다.'

In [12]:
# %pip install -U langchain langchain_classic langchainhub --quiet

In [13]:
from langchain_classic import hub

prompt = hub.pull('rlm/rag-prompt')


In [14]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [15]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={'prompt': prompt}
)

In [16]:
ai_message = qa_chain.invoke({'query': query})

In [17]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '컨텍스트에 소득세 계산을 위한 구체적인 세율이나 기준은 포함되어 있지 않습니다. 따라서 연봉 5천만 원에 대한 소득세를 계산하기 위해서는 외부의 한국 소득세 관련 자료를 참고해야 합니다. 일반적으로 소득세는 연간 소득 금액에 따라 달라지며, 다양한 공제 항목도 고려합니다.'}